# DeepDGA — corrected training notebook

This version preserves the original model layers and dimensions. It fixes reproducibility, split overlap, padded-sequence handling, best-checkpoint selection, artifact persistence, and row-aligned family evaluation.

Before running, update `DRIVE_PROJECT_DIR` if your Google Drive folder has a different name.

In [ ]:
%pip install -q wordninja gensim joblib scikit-learn pandas tqdm

In [ ]:
from pathlib import Path
import ipaddress
import json
import math
import os
import random
import string
import time
from urllib.parse import urlsplit

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wordninja
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42

def set_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('PyTorch:', torch.__version__)

Device: cuda
PyTorch: 2.11.0+cu130


In [ ]:
# Mount Google Drive and select the project directory.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/DeepDGA')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    if not DRIVE_PROJECT_DIR.is_dir():
        raise FileNotFoundError(
            f'Project directory not found: {DRIVE_PROJECT_DIR}. '
            'Update DRIVE_PROJECT_DIR in this cell.'
        )
    os.chdir(DRIVE_PROJECT_DIR)
except ImportError:
    print('Not running on Colab; using current directory.')

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data' if (PROJECT_DIR / 'data').is_dir() else PROJECT_DIR
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / 'deepdga_fixed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train_genAI.csv'
VAL_PATH = DATA_DIR / 'val_genAI.csv'
TEST_PATH = DATA_DIR / 'test_genAI.csv'

MODEL_PATH = OUTPUT_DIR / 'deepDGA_best_bundle.pt'
W2V_PATH = OUTPUT_DIR / 'deepDGA_word2vec.model'
TFIDF_PATH = OUTPUT_DIR / 'deepDGA_tfidf.joblib'
HISTORY_PATH = OUTPUT_DIR / 'training_history.csv'

for path in (TRAIN_PATH, VAL_PATH, TEST_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)

print('Project:', PROJECT_DIR)
print('Data:', DATA_DIR)
print('Outputs:', OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/DeepDGA
Data: /content/drive/MyDrive/DeepDGA/data
Outputs: /content/drive/MyDrive/DeepDGA/artifacts/deepdga_fixed


In [ ]:
ALPHABET = string.ascii_lowercase + string.digits + '.' + '-'
CHAR2IDX = {character: index + 1 for index, character in enumerate(ALPHABET)}
VOCAB_SIZE = len(CHAR2IDX) + 1
MAX_LEN = 75
THRESHOLD = 0.5

def normalize_domain(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError('Domain must be a non-empty string')
    raw = value.strip()
    parsed = urlsplit(raw if '://' in raw else '//' + raw)
    if parsed.scheme and parsed.scheme.lower() not in ('http', 'https'):
        raise ValueError(f'Unsupported URL scheme: {parsed.scheme}')
    if parsed.username is not None or parsed.password is not None:
        raise ValueError('Credentials are not allowed in domain input')
    host = (parsed.hostname or '').rstrip('.').lower()
    if not host:
        raise ValueError('Hostname is empty')
    try:
        ipaddress.ip_address(host)
    except ValueError:
        pass
    else:
        raise ValueError('IP address is not a domain')
    host = host.encode('idna').decode('ascii')
    if len(host) > 253:
        raise ValueError('Hostname exceeds 253 characters')
    invalid = sorted(set(host) - set(ALPHABET))
    if invalid:
        raise ValueError(f'Unsupported hostname characters: {invalid}')
    return host

def load_and_clean(path, split_name):
    frame = pd.read_csv(path, low_memory=False)
    required = {'domain_name', 'label'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{split_name}: missing columns {sorted(missing)}')
    if frame['domain_name'].isna().any() or frame['label'].isna().any():
        raise ValueError(f'{split_name}: missing domain or label')
    if not frame['label'].isin([0, 1]).all():
        raise ValueError(f'{split_name}: label must contain only 0 and 1')

    normalized = []
    invalid_examples = []
    for row_index, value in enumerate(tqdm(frame['domain_name'], desc=f'Clean {split_name}')):
        try:
            normalized.append(normalize_domain(value))
        except (TypeError, ValueError, UnicodeError) as error:
            normalized.append(None)
            if len(invalid_examples) < 10:
                invalid_examples.append({
                    'row': row_index, 'value': str(value), 'error': str(error)
                })
    frame = frame.copy()
    frame['normalized_domain'] = normalized
    invalid_count = int(frame['normalized_domain'].isna().sum())
    frame = frame.dropna(subset=['normalized_domain']).copy()

    conflicts = frame.groupby('normalized_domain')['label'].nunique()
    conflict_domains = set(conflicts[conflicts > 1].index)
    conflict_mask = frame['normalized_domain'].isin(conflict_domains)
    conflicting_rows_removed = int(conflict_mask.sum())
    conflict_examples = sorted(conflict_domains)[:10]
    # A domain carrying both labels has no reliable target. Remove every
    # occurrence instead of selecting a label based on row order.
    frame = frame.loc[~conflict_mask].copy()

    duplicate_count = int(frame['normalized_domain'].duplicated().sum())
    frame = frame.drop_duplicates('normalized_domain', keep='first').reset_index(drop=True)
    return frame, {
        'invalid_removed': invalid_count,
        'invalid_examples': invalid_examples,
        'conflicting_domains_removed': len(conflict_domains),
        'conflicting_rows_removed': conflicting_rows_removed,
        'conflict_examples': conflict_examples,
        'duplicates_removed': duplicate_count,
    }

In [ ]:
train_df, train_report = load_and_clean(TRAIN_PATH, 'train')
val_df, val_report = load_and_clean(VAL_PATH, 'validation')
test_df, test_report = load_and_clean(TEST_PATH, 'test')

# Preserve train first, then remove leakage from validation and test.
train_keys = set(train_df['normalized_domain'])
val_overlap = val_df['normalized_domain'].isin(train_keys)
val_report['train_overlap_removed'] = int(val_overlap.sum())
val_df = val_df.loc[~val_overlap].reset_index(drop=True)

seen = train_keys | set(val_df['normalized_domain'])
test_overlap = test_df['normalized_domain'].isin(seen)
test_report['earlier_split_overlap_removed'] = int(test_overlap.sum())
test_df = test_df.loc[~test_overlap].reset_index(drop=True)

for split_name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    if set(frame['label'].unique()) != {0, 1}:
        raise ValueError(f'{split_name}: both classes are required after cleaning')

data_report = {
    'train': {**train_report, 'rows': len(train_df)},
    'validation': {**val_report, 'rows': len(val_df)},
    'test': {**test_report, 'rows': len(test_df)},
}
(OUTPUT_DIR / 'data_cleaning_report.json').write_text(
    json.dumps(data_report, indent=2, ensure_ascii=False), encoding='utf-8'
)
print(json.dumps(data_report, indent=2, ensure_ascii=False))
print('Class counts:')
for name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print(name, frame['label'].value_counts().sort_index().to_dict())

Clean train:   0%|          | 0/986316 [00:00<?, ?it/s]

Clean validation:   0%|          | 0/123290 [00:00<?, ?it/s]

Clean test:   0%|          | 0/123293 [00:00<?, ?it/s]

{
  "train": {
    "invalid_removed": 8,
    "invalid_examples": [
      {
        "row": 970384,
        "value": "ssasass_soaanno.icu",
        "error": "Unsupported hostname characters: ['_']"
      },
      {
        "row": 973738,
        "value": "assissseaisa_c.icu",
        "error": "Unsupported hostname characters: ['_']"
      },
      {
        "row": 973815,
        "value": "arsesus_doaannos.icu",
        "error": "Unsupported hostname characters: ['_']"
      },
      {
        "row": 978495,
        "value": "sssusesasesa_nou.icu",
        "error": "Unsupported hostname characters: ['_']"
      },
      {
        "row": 978605,
        "value": "sssi_ssiaosnnn.icu",
        "error": "Unsupported hostname characters: ['_']"
      },
      {
        "row": 982899,
        "value": "aaaesase_esavnou.icu",
        "error": "Unsupported hostname characters: ['_']"
      },
      {
        "row": 984077,
        "value": "sadiassadisanno_.icu",
        "error": "Unsupported ho

In [ ]:
def domain_to_char_tensor(domain):
    text = domain[:MAX_LEN]
    values = [CHAR2IDX[character] for character in text]
    length = len(values)
    values += [0] * (MAX_LEN - length)
    return torch.tensor(values, dtype=torch.long), length

def domain_to_words(domain):
    tokens = []
    for part in domain.split('.'):
        tokens.extend(wordninja.split(part))
    return tokens

train_sentences = [domain_to_words(domain) for domain in tqdm(
    train_df['normalized_domain'], desc='Word segmentation'
)]

w2v = Word2Vec(
    sentences=train_sentences,
    vector_size=100,
    window=5,
    sg=1,
    min_count=1,
    workers=1,
    seed=SEED,
)
tfidf = TfidfVectorizer()
tfidf.fit([' '.join(tokens) for tokens in train_sentences])

# These artifacts are required to reproduce the 500-dimensional word input.
w2v.save(str(W2V_PATH))
joblib.dump(tfidf, TFIDF_PATH)
print('Saved:', W2V_PATH)
print('Saved:', TFIDF_PATH)

Word segmentation:   0%|          | 0/986306 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/DeepDGA/artifacts/deepdga_fixed/deepDGA_word2vec.model
Saved: /content/drive/MyDrive/DeepDGA/artifacts/deepdga_fixed/deepDGA_tfidf.joblib


In [ ]:
def aggregate_domain(tokens):
    vectors = []
    weights = []
    for token in tokens:
        if token not in w2v.wv:
            continue
        vectors.append(w2v.wv[token])
        if token in tfidf.vocabulary_:
            weights.append(tfidf.idf_[tfidf.vocabulary_[token]])
        else:
            weights.append(1.0)
    if not vectors:
        return np.zeros(500, dtype=np.float32)

    vectors = np.asarray(vectors, dtype=np.float32)
    weights = np.asarray(weights, dtype=np.float32)
    features = [
        np.min(vectors, axis=0),
        np.mean(vectors, axis=0),
        np.max(vectors, axis=0),
        np.sum(vectors, axis=0),
        np.average(vectors, axis=0, weights=weights),
    ]
    return np.concatenate(features).astype(np.float32, copy=False)

class DomainDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        domain = row['normalized_domain']
        x_char, char_length = domain_to_char_tensor(domain)
        x_word = torch.tensor(
            aggregate_domain(domain_to_words(domain)), dtype=torch.float32
        )
        return {
            'x_char': x_char,
            'char_length': torch.tensor(char_length, dtype=torch.long),
            'x_word': x_word,
            'label': torch.tensor(float(row['label']), dtype=torch.float32),
            'row_index': torch.tensor(index, dtype=torch.long),
        }

train_dataset = DomainDataset(train_df)
val_dataset = DomainDataset(val_df)
test_dataset = DomainDataset(test_df)

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, generator=loader_generator, num_workers=0
)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)
print(len(train_dataset), len(val_dataset), len(test_dataset))

986306 123289 123291


In [ ]:
class DeepDGA(nn.Module):
    # Layer structure and dimensions are unchanged from deepDGA_ver2.
    def __init__(self, vocab_size):
        super().__init__()
        self.char_embedding = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=128, padding_idx=0
        )
        self.char_lstm = nn.LSTM(
            input_size=128, hidden_size=128, batch_first=True, bidirectional=True
        )
        self.word_fc = nn.Linear(500, 128)
        self.fc1 = nn.Linear(256 + 128, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x_char, x_word, char_lengths):
        embedded = self.char_embedding(x_char)
        packed = pack_padded_sequence(
            embedded,
            char_lengths.detach().cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, (hidden, _) = self.char_lstm(packed)
        h_forward = hidden[0]
        h_backward = hidden[1]
        z_char = torch.cat([h_forward, h_backward], dim=1)
        z_word = self.relu(self.word_fc(x_word))
        z = torch.cat([z_char, z_word], dim=1)
        z = self.dropout(z)
        z = torch.relu(self.fc1(z))
        return torch.sigmoid(self.fc2(z)).squeeze(1)

model_for_count = DeepDGA(VOCAB_SIZE)
total_params = sum(parameter.numel() for parameter in model_for_count.parameters())
trainable_params = sum(
    parameter.numel() for parameter in model_for_count.parameters() if parameter.requires_grad
)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
assert total_params == 358_017, 'The original model structure changed unexpectedly'

Total parameters: 358,017
Trainable parameters: 358,017


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_samples = 0
    for batch in tqdm(loader, desc='train', leave=False):
        x_char = batch['x_char'].to(device)
        lengths = batch['char_length'].to(device)
        x_word = batch['x_word'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad(set_to_none=True)
        probabilities = model(x_char, x_word, lengths)
        loss = criterion(probabilities, labels)
        if not torch.isfinite(loss):
            raise RuntimeError('Non-finite training loss')
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        total_samples += len(labels)
    return total_loss / total_samples

@torch.inference_mode()
def evaluate(model, loader, criterion, device, threshold=THRESHOLD):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    all_probabilities, all_labels, all_indices = [], [], []
    for batch in tqdm(loader, desc='eval', leave=False):
        x_char = batch['x_char'].to(device)
        lengths = batch['char_length'].to(device)
        x_word = batch['x_word'].to(device)
        labels = batch['label'].to(device)
        probabilities = model(x_char, x_word, lengths)
        loss = criterion(probabilities, labels)
        total_loss += loss.item() * len(labels)
        total_samples += len(labels)
        all_probabilities.extend(probabilities.cpu().tolist())
        all_labels.extend(labels.cpu().to(torch.int64).tolist())
        all_indices.extend(batch['row_index'].tolist())

    y_true = np.asarray(all_labels, dtype=np.int64)
    y_score = np.asarray(all_probabilities, dtype=np.float64)
    y_pred = (y_score >= threshold).astype(np.int64)
    metrics = {
        'loss': float(total_loss / total_samples),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, y_score)),
        'average_precision': float(average_precision_score(y_true, y_score)),
    }
    return metrics, y_true, y_score, y_pred, np.asarray(all_indices, dtype=np.int64)

def save_checkpoint(model, epoch, validation_metrics):
    checkpoint = {
        'format_version': 1,
        'model_name': 'DeepDGA_fixed',
        'epoch': int(epoch),
        'threshold': float(THRESHOLD),
        'model_state_dict': {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        },
        'model_config': {
            'vocab_size': VOCAB_SIZE,
            'char_embedding_dim': 128,
            'char_lstm_hidden': 128,
            'word_feature_dim': 500,
            'word_hidden_dim': 128,
            'fusion_hidden_dim': 64,
            'dropout': 0.5,
        },
        'preprocessing': {
            'alphabet': ALPHABET,
            'char2idx': CHAR2IDX,
            'max_len': MAX_LEN,
            'threshold': THRESHOLD,
            'normalization': 'urlsplit-hostname-idna-lower-v1',
            'word2vec_file': W2V_PATH.name,
            'tfidf_file': TFIDF_PATH.name,
        },
        'label_mapping': {'0': 'Benign', '1': 'DGA'},
        'validation_metrics': validation_metrics,
        'seed': SEED,
    }
    torch.save(checkpoint, MODEL_PATH)

In [ ]:
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3
EARLY_STOPPING_PATIENCE = 3

model = DeepDGA(VOCAB_SIZE).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCELoss()
best_validation_f1 = -math.inf
epochs_without_improvement = 0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    started = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics, _, _, _, _ = evaluate(model, val_loader, criterion, device)
    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        **{f'val_{key}': value for key, value in val_metrics.items()},
        'epoch_seconds': time.perf_counter() - started,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | acc={val_metrics['accuracy']:.4f} | "
        f"precision={val_metrics['precision']:.4f} | recall={val_metrics['recall']:.4f} | "
        f"f1={val_metrics['f1']:.4f}"
    )

    if val_metrics['f1'] > best_validation_f1:
        best_validation_f1 = val_metrics['f1']
        epochs_without_improvement = 0
        save_checkpoint(model, epoch, val_metrics)
        print('  Saved new best checkpoint:', MODEL_PATH)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print('Early stopping: validation F1 did not improve.')
            break

print('Best validation F1:', best_validation_f1)

train:   0%|          | 0/30823 [00:00<?, ?it/s]

eval:   0%|          | 0/3853 [00:00<?, ?it/s]

Epoch 01/10 | train_loss=0.0870 | val_loss=0.0594 | acc=0.9799 | precision=0.9733 | recall=0.9486 | f1=0.9608
  Saved new best checkpoint: /content/drive/MyDrive/DeepDGA/artifacts/deepdga_fixed/deepDGA_best_bundle.pt


train:   0%|          | 0/30823 [00:00<?, ?it/s]

eval:   0%|          | 0/3853 [00:00<?, ?it/s]

...

eval:   0%|          | 0/3853 [00:00<?, ?it/s]

Epoch 06/10 | train_loss=0.0537 | val_loss=0.0502 | acc=0.9833 | precision=0.9788 | recall=0.9564 | f1=0.9675


train:   0%|          | 0/30823 [00:00<?, ?it/s]

eval:   0%|          | 0/3853 [00:00<?, ?it/s]

Epoch 07/10 | train_loss=0.0526 | val_loss=0.0485 | acc=0.9836 | precision=0.9842 | recall=0.9521 | f1=0.9679
Early stopping: validation F1 did not improve.
Best validation F1: 0.967916127198661


In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
if checkpoint.get('format_version') != 1:
    raise ValueError('Unsupported checkpoint format')
if checkpoint.get('label_mapping') != {'0': 'Benign', '1': 'DGA'}:
    raise ValueError('Invalid label mapping')

model = DeepDGA(checkpoint['model_config']['vocab_size'])
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.to(device).eval()

test_metrics, y_true, y_score, y_pred, row_indices = evaluate(
    model, test_loader, criterion, device, threshold=float(checkpoint['threshold'])
)
print(json.dumps(test_metrics, indent=2))
print(classification_report(
    y_true, y_pred, target_names=['Benign', 'DGA'], digits=4, zero_division=0
))

# row_indices keeps predictions aligned even if loader settings change later.
test_predictions = test_df.iloc[row_indices].copy().reset_index(drop=True)
test_predictions['truth'] = y_true
test_predictions['prediction'] = y_pred
test_predictions['score'] = y_score
test_predictions['model'] = checkpoint['model_name']
test_predictions.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
(OUTPUT_DIR / 'test_metrics.json').write_text(
    json.dumps(test_metrics, indent=2), encoding='utf-8'
)

eval:   0%|          | 0/3853 [00:00<?, ?it/s]

{
  "loss": 0.04827119366958348,
  "accuracy": 0.982723799790739,
  "precision": 0.9779028492852803,
  "recall": 0.9550579343514788,
  "f1": 0.9663453942170959,
  "roc_auc": 0.9974721220517538,
  "average_precision": 0.994459302877684
}
              precision    recall  f1-score   support

      Benign     0.9844    0.9924    0.9884     91272
         DGA     0.9779    0.9551    0.9663     32019

    accuracy                         0.9827    123291
   macro avg     0.9811    0.9737    0.9774    123291
weighted avg     0.9827    0.9827    0.9827    123291



236

In [ ]:
def evaluate_by_family(predictions):
    if 'subclass' not in predictions.columns:
        print('No subclass column; family report skipped.')
        return pd.DataFrame()

    rows = []
    for family, group in predictions.groupby('subclass', dropna=False):
        truth = group['truth'].to_numpy(dtype=np.int64)
        predicted = group['prediction'].to_numpy(dtype=np.int64)
        scores = group['score'].to_numpy(dtype=np.float64)
        tn, fp, fn, tp = confusion_matrix(truth, predicted, labels=[0, 1]).ravel()
        has_both_classes = len(np.unique(truth)) == 2
        dga_count = int((truth == 1).sum())
        benign_count = int((truth == 0).sum())
        rows.append({
            'family': family,
            'n_samples': len(group),
            'benign_count': benign_count,
            'dga_count': dga_count,
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
            'accuracy': float(accuracy_score(truth, predicted)),
            # Precision/F1 and AUC are not reported for one-class families.
            'precision': float(precision_score(truth, predicted, zero_division=0))
                if has_both_classes else np.nan,
            'recall': float(recall_score(truth, predicted, zero_division=0))
                if dga_count else np.nan,
            'f1': float(f1_score(truth, predicted, zero_division=0))
                if has_both_classes else np.nan,
            'roc_auc': float(roc_auc_score(truth, scores))
                if has_both_classes else np.nan,
            'pr_auc': float(average_precision_score(truth, scores))
                if has_both_classes else np.nan,
            'dga_detection_rate': float(tp / (tp + fn)) if dga_count else np.nan,
            'benign_false_positive_rate': float(fp / (tn + fp))
                if benign_count else np.nan,
            'mean_score': float(scores.mean()),
        })

    report = pd.DataFrame(rows).sort_values('n_samples', ascending=False)
    report.to_csv(OUTPUT_DIR / 'deepDGA_test_family_metrics.csv', index=False)
    return report

family_report = evaluate_by_family(test_predictions)
display(family_report.head(30))

,family,n_samples,benign_count,dga_count,tn,fp,fn,tp,accuracy,precision,recall,f1,roc_auc,pr_auc,dga_detection_rate,benign_false_positive_rate,mean_score
1,benign,91272,91272,0,90581,691,0,0,0.992429,NaN,NaN,NaN,NaN,NaN,NaN,0.007571,0.016512
3,ccleaner,7956,0,7956,0,0,1,7955,0.999874,NaN,0.999874,NaN,NaN,NaN,0.999874,NaN,0.999875
26,qadars,2951,0,2951,0,0,6,2945,0.997967,NaN,0.997967,NaN,NaN,NaN,0.997967,NaN,0.995005
10,gozi,1963,0,1963,0,0,178,1785,0.909322,NaN,0.909322,NaN,NaN,NaN,0.909322,NaN,0.886681
34,suppobox,1498,0,1498,0,0,7,1491,0.995327,NaN,0.995327,NaN,NaN,NaN,0.995327,NaN,0.979411
5,deepdga,1200,0,1200,0,0,1,1199,0.999167,NaN,0.999167,NaN,NaN,NaN,0.999167,NaN,0.999146
11,improved_pykspa,1013,0,1013,0,0,125,888,0.876604,NaN,0.876604,NaN,NaN,NaN,0.876604,NaN,0.832216
37,titan,1000,0,1000,0,0,524,476,0.476000,NaN,0.476000,NaN,NaN,NaN,0.476000,NaN,0.499457
2,bigviktor,820,0,820,0,0,87,733,0.893902,NaN,0.893902,NaN,NaN,NaN,0.893902,NaN,0.881723
8,enviserv,812,0,812,0,0,7,805,0.991379,NaN,0.991379,NaN,NaN,NaN,0.991379,NaN,0.986522


In [ ]:
# Verify that all deployment artifacts can be loaded independently.
loaded_w2v = Word2Vec.load(str(W2V_PATH))
loaded_tfidf = joblib.load(TFIDF_PATH)
loaded_checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)

assert loaded_checkpoint['model_config']['vocab_size'] == VOCAB_SIZE
assert loaded_checkpoint['threshold'] == THRESHOLD
assert loaded_checkpoint['preprocessing']['char2idx'] == CHAR2IDX
assert loaded_w2v.vector_size == 100
assert hasattr(loaded_tfidf, 'idf_')

print('Deployment artifacts verified:')
for path in (MODEL_PATH, W2V_PATH, TFIDF_PATH, HISTORY_PATH):
    print(f'  {path.name}: {path.stat().st_size / (1024 * 1024):.2f} MiB')

Deployment artifacts verified:
  deepDGA_best_bundle.pt: 1.37 MiB
  deepDGA_word2vec.model: 73.72 MiB
  deepDGA_tfidf.joblib: 1.87 MiB
  training_history.csv: 0.00 MiB
